# Modelação Silver → Gold

POC de dados de energia no Microsoft Fabric, desenvolvida no âmbito da preparação para a certificação DP-700.

Este notebook cria dimensões e tabelas de factos segundo um modelo em estrela, gera chaves substitutas, trata registos sem correspondência e grava as estruturas analíticas em formato Delta.

> Os dados utilizados são inteiramente fictícios.


## 1. Carregar bibliotecas e tabelas Silver


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

clientes = spark.table("dbo.silver_clientes")
unidades = spark.table("dbo.silver_unidades")
consumos = spark.table("dbo.silver_consumos")
faturas = spark.table("dbo.silver_faturas")
pedidos = spark.table("dbo.silver_pedidos")

print("Clientes:", clientes.count())
print("Unidades:", unidades.count())
print("Consumos:", consumos.count())
print("Faturas:", faturas.count())
print("Pedidos:", pedidos.count())


## 2. Criar a dimensão Cliente


In [ ]:
dim_cliente = (
    clientes
    .select(
        F.abs(F.xxhash64("ClienteID"))
            .cast("long")
            .alias("ClienteKey"),
        "ClienteID",
        "NomeCliente",
        "Concelho",
        "Segmento",
        "DataAdesao",
        "EstadoCliente"
    )
)

(
    dim_cliente.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_dim_cliente")
)

print("Dimensão Cliente criada:", dim_cliente.count())
display(dim_cliente)


## 3. Criar a dimensão Unidade


In [ ]:
dim_unidade = (
    unidades
    .join(
        dim_cliente.select(
            "ClienteID",
            "ClienteKey"
        ),
        "ClienteID",
        "left"
    )
    .select(
        F.abs(F.xxhash64("UnidadeID"))
            .cast("long")
            .alias("UnidadeKey"),

        "UnidadeID",
        "ClienteKey",
        "ClienteID",
        "TipoUnidade",
        "Concelho",
        "EstadoUnidade",
        "DataAtivacao"
    )
)

(
    dim_unidade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_dim_unidade")
)

print("Dimensão Unidade criada:", dim_unidade.count())

print(
    "Unidades sem ClienteKey:",
    dim_unidade.filter(F.col("ClienteKey").isNull()).count()
)

display(dim_unidade)


## 4. Criar a dimensão Data


In [ ]:
# Reunir todas as datas utilizadas no modelo
datas_modelo = (
    consumos.select(F.to_date("DataLeitura").alias("Data"))
    .union(
        faturas.select(F.to_date("DataEmissao").alias("Data"))
    )
    .union(
        faturas.select(F.to_date("DataVencimento").alias("Data"))
    )
    .union(
        faturas.select(F.to_date("DataPagamento").alias("Data"))
    )
    .union(
        pedidos.select(F.to_date("DataAbertura").alias("Data"))
    )
    .union(
        pedidos.select(F.to_date("DataConclusao").alias("Data"))
    )
    .filter(F.col("Data").isNotNull())
)

# Determinar o intervalo do calendário
limites = (
    datas_modelo
    .agg(
        F.min("Data").alias("DataMinima"),
        F.max("Data").alias("DataMaxima")
    )
    .first()
)

data_minima = limites["DataMinima"]
data_maxima = limites["DataMaxima"]

print("Data mínima:", data_minima)
print("Data máxima:", data_maxima)


In [ ]:
# Criar uma linha para cada dia
dim_data = spark.sql(
    f"""
    SELECT EXPLODE(
        SEQUENCE(
            TO_DATE('{data_minima}'),
            TO_DATE('{data_maxima}'),
            INTERVAL 1 DAY
        )
    ) AS Data
    """
)

meses = F.array(
    F.lit("Janeiro"),
    F.lit("Fevereiro"),
    F.lit("Março"),
    F.lit("Abril"),
    F.lit("Maio"),
    F.lit("Junho"),
    F.lit("Julho"),
    F.lit("Agosto"),
    F.lit("Setembro"),
    F.lit("Outubro"),
    F.lit("Novembro"),
    F.lit("Dezembro")
)

dias_semana = F.array(
    F.lit("Domingo"),
    F.lit("Segunda-feira"),
    F.lit("Terça-feira"),
    F.lit("Quarta-feira"),
    F.lit("Quinta-feira"),
    F.lit("Sexta-feira"),
    F.lit("Sábado")
)

dim_data = (
    dim_data
    .withColumn(
        "DataKey",
        F.date_format("Data", "yyyyMMdd").cast("int")
    )
    .withColumn("Ano", F.year("Data"))
    .withColumn("Trimestre", F.quarter("Data"))
    .withColumn("MesNumero", F.month("Data"))
    .withColumn(
        "MesNome",
        F.element_at(meses, F.month("Data"))
    )
    .withColumn("AnoMes", F.date_format("Data", "yyyy-MM"))
    .withColumn("Dia", F.dayofmonth("Data"))
    .withColumn(
        "DiaSemanaNumero",
        F.dayofweek("Data")
    )
    .withColumn(
        "DiaSemanaNome",
        F.element_at(dias_semana, F.dayofweek("Data"))
    )
    .withColumn(
        "FimDeSemana",
        F.dayofweek("Data").isin(1, 7)
    )
    .select(
        "DataKey",
        "Data",
        "Ano",
        "Trimestre",
        "MesNumero",
        "MesNome",
        "AnoMes",
        "Dia",
        "DiaSemanaNumero",
        "DiaSemanaNome",
        "FimDeSemana"
    )
)

(
    dim_data.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_dim_data")
)

print("Dimensão Data criada:", dim_data.count())
display(dim_data.orderBy("Data"))


## 5. Criar o facto Consumo


In [ ]:
# Garantir a utilização da dimensão gravada
dim_unidade_gold = spark.table("dbo.gold_dim_unidade")

fact_consumo = (
    consumos
    .join(
        dim_unidade_gold.select(
            "UnidadeID",
            "UnidadeKey",
            "ClienteKey"
        ),
        "UnidadeID",
        "left"
    )
    .withColumn(
        "DataLeituraKey",
        F.date_format(
            F.to_date("DataLeitura"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "ConsumoID",
        "UnidadeKey",
        "ClienteKey",
        "DataLeituraKey",
        "UnidadeID",
        "DataLeitura",
        "Periodo",
        "QuantidadeKWh",
        "TipoLeitura"
    )
)

# Validar antes da gravação
print("Total de consumos:", fact_consumo.count())

print(
    "Consumos sem UnidadeKey:",
    fact_consumo
    .filter(F.col("UnidadeKey").isNull())
    .count()
)

print(
    "Consumos sem ClienteKey:",
    fact_consumo
    .filter(F.col("ClienteKey").isNull())
    .count()
)

print(
    "Consumos sem DataLeituraKey:",
    fact_consumo
    .filter(F.col("DataLeituraKey").isNull())
    .count()
)


### Identificar e separar consumos rejeitados


In [ ]:
consumos_com_problemas = (
    fact_consumo
    .filter(
        F.col("UnidadeKey").isNull()
        | F.col("ClienteKey").isNull()
        | F.col("DataLeituraKey").isNull()
    )
)

display(
    consumos_com_problemas.select(
        "ConsumoID",
        "UnidadeID",
        "DataLeitura",
        "Periodo",
        "QuantidadeKWh",
        "TipoLeitura",
        "UnidadeKey",
        "ClienteKey",
        "DataLeituraKey"
    )
)


In [ ]:
# Consumos válidos
fact_consumo_valido = (
    fact_consumo
    .filter(
        F.col("UnidadeKey").isNotNull()
        & F.col("ClienteKey").isNotNull()
        & F.col("DataLeituraKey").isNotNull()
    )
)

# Consumos rejeitados, com indicação do motivo
fact_consumo_rejeitado = (
    fact_consumo
    .filter(
        F.col("UnidadeKey").isNull()
        | F.col("ClienteKey").isNull()
        | F.col("DataLeituraKey").isNull()
    )
    .withColumn(
        "MotivoRejeicao",
        F.concat_ws(
            "; ",
            F.when(
                F.col("UnidadeKey").isNull(),
                F.lit("UnidadeID inexistente")
            ),
            F.when(
                F.col("ClienteKey").isNull(),
                F.lit("Cliente não identificado")
            ),
            F.when(
                F.col("DataLeituraKey").isNull(),
                F.lit("DataLeitura nula ou inválida")
            )
        )
    )
    .withColumn(
        "DataRejeicao",
        F.current_timestamp()
    )
)

print("Consumos válidos:", fact_consumo_valido.count())
print("Consumos rejeitados:", fact_consumo_rejeitado.count())

display(fact_consumo_rejeitado)


In [ ]:
(
    fact_consumo_valido.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_fact_consumo")
)

(
    fact_consumo_rejeitado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_fact_consumo_rejeitados")
)

print(
    "gold_fact_consumo:",
    spark.table("dbo.gold_fact_consumo").count()
)

print(
    "gold_fact_consumo_rejeitados:",
    spark.table("dbo.gold_fact_consumo_rejeitados").count()
)


## 6. Criar o facto Faturação


In [ ]:
dim_cliente_gold = spark.table("dbo.gold_dim_cliente")

fact_faturacao = (
    faturas
    .join(
        dim_cliente_gold.select(
            "ClienteID",
            "ClienteKey"
        ),
        "ClienteID",
        "left"
    )
    .withColumn(
        "DataEmissaoKey",
        F.date_format(
            F.to_date("DataEmissao"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "DataVencimentoKey",
        F.date_format(
            F.to_date("DataVencimento"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "DataPagamentoKey",
        F.when(
            F.col("DataPagamento").isNotNull(),
            F.date_format(
                F.to_date("DataPagamento"),
                "yyyyMMdd"
            ).cast("int")
        )
    )
    .withColumn(
        "DiasAtePagamento",
        F.when(
            F.col("DataPagamento").isNotNull(),
            F.datediff("DataPagamento", "DataEmissao")
        )
    )
    .withColumn(
        "Pago",
        F.when(
            F.col("EstadoPagamento") == "Pago",
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        "FaturaID",
        "ClienteKey",
        "DataEmissaoKey",
        "DataVencimentoKey",
        "DataPagamentoKey",
        "ClienteID",
        "Periodo",
        "DataEmissao",
        "DataVencimento",
        "DataPagamento",
        "ValorSemIVA",
        "ValorIVA",
        "ValorTotal",
        "EstadoPagamento",
        "Pago",
        "DiasAtePagamento"
    )
)


In [ ]:
print("Total de faturas:", fact_faturacao.count())

print(
    "Faturas sem ClienteKey:",
    fact_faturacao
    .filter(F.col("ClienteKey").isNull())
    .count()
)

print(
    "Faturas sem DataEmissaoKey:",
    fact_faturacao
    .filter(F.col("DataEmissaoKey").isNull())
    .count()
)

print(
    "Faturas sem DataVencimentoKey:",
    fact_faturacao
    .filter(F.col("DataVencimentoKey").isNull())
    .count()
)

print(
    "Faturas sem ValorTotal:",
    fact_faturacao
    .filter(F.col("ValorTotal").isNull())
    .count()
)


In [ ]:
(
    fact_faturacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_fact_faturacao")
)

print(
    "Tabela gold_fact_faturacao criada:",
    spark.table("dbo.gold_fact_faturacao").count()
)

display(
    spark.table("dbo.gold_fact_faturacao")
)


## 7. Criar o facto Pedido


In [ ]:
dim_unidade_gold = spark.table("dbo.gold_dim_unidade")

fact_pedido = (
    pedidos
    .join(
        dim_unidade_gold.select(
            "UnidadeID",
            "UnidadeKey",
            "ClienteKey"
        ),
        "UnidadeID",
        "left"
    )
    .withColumn(
        "DataAberturaKey",
        F.date_format(
            F.to_date("DataAbertura"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "DataConclusaoKey",
        F.when(
            F.col("DataConclusao").isNotNull(),
            F.date_format(
                F.to_date("DataConclusao"),
                "yyyyMMdd"
            ).cast("int")
        )
    )
    .withColumn(
        "DuracaoHoras",
        F.when(
            F.col("DataConclusao").isNotNull(),
            F.round(
                (
                    F.unix_timestamp("DataConclusao")
                    - F.unix_timestamp("DataAbertura")
                ) / 3600,
                2
            )
        )
    )
    .withColumn(
        "Concluido",
        F.when(
            F.col("DataConclusao").isNotNull(),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        "PedidoID",
        "UnidadeKey",
        "ClienteKey",
        "DataAberturaKey",
        "DataConclusaoKey",
        "UnidadeID",
        "DataAbertura",
        "DataConclusao",
        "Canal",
        "Estado",
        "Prioridade",
        "Servico",
        "Concluido",
        "DuracaoHoras"
    )
)


In [ ]:
print("Total de pedidos:", fact_pedido.count())

print(
    "Pedidos sem UnidadeKey:",
    fact_pedido
    .filter(F.col("UnidadeKey").isNull())
    .count()
)

print(
    "Pedidos sem ClienteKey:",
    fact_pedido
    .filter(F.col("ClienteKey").isNull())
    .count()
)

print(
    "Pedidos sem DataAberturaKey:",
    fact_pedido
    .filter(F.col("DataAberturaKey").isNull())
    .count()
)


### Identificar e separar pedidos rejeitados


In [ ]:
pedidos_com_problemas = (
    fact_pedido
    .filter(
        F.col("UnidadeKey").isNull()
        | F.col("ClienteKey").isNull()
        | F.col("DataAberturaKey").isNull()
    )
)

display(
    pedidos_com_problemas.select(
        "PedidoID",
        "UnidadeID",
        "DataAbertura",
        "DataConclusao",
        "Estado",
        "Prioridade",
        "Servico",
        "UnidadeKey",
        "ClienteKey",
        "DataAberturaKey"
    )
)


In [ ]:
# Separar pedidos válidos
fact_pedido_valido = (
    fact_pedido
    .filter(
        F.col("UnidadeKey").isNotNull()
        & F.col("ClienteKey").isNotNull()
        & F.col("DataAberturaKey").isNotNull()
    )
)

# Separar pedidos rejeitados
fact_pedido_rejeitado = (
    fact_pedido
    .filter(
        F.col("UnidadeKey").isNull()
        | F.col("ClienteKey").isNull()
        | F.col("DataAberturaKey").isNull()
    )
    .withColumn(
        "MotivoRejeicao",
        F.concat_ws(
            "; ",
            F.when(
                F.col("UnidadeKey").isNull(),
                F.lit("UnidadeID inexistente")
            ),
            F.when(
                F.col("ClienteKey").isNull(),
                F.lit("Cliente não identificado")
            ),
            F.when(
                F.col("DataAberturaKey").isNull(),
                F.lit("DataAbertura nula ou inválida")
            )
        )
    )
    .withColumn(
        "DataRejeicao",
        F.current_timestamp()
    )
)

print("Pedidos válidos:", fact_pedido_valido.count())
print("Pedidos rejeitados:", fact_pedido_rejeitado.count())


In [ ]:
(
    fact_pedido_valido.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_fact_pedido")
)

(
    fact_pedido_rejeitado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.gold_fact_pedido_rejeitados")
)

print(
    "gold_fact_pedido:",
    spark.table("dbo.gold_fact_pedido").count()
)

print(
    "gold_fact_pedido_rejeitados:",
    spark.table("dbo.gold_fact_pedido_rejeitados").count()
)


## 8. Validação final da camada Gold


In [ ]:
tabelas_gold = [
    "dbo.gold_dim_cliente",
    "dbo.gold_dim_unidade",
    "dbo.gold_dim_data",
    "dbo.gold_fact_consumo",
    "dbo.gold_fact_consumo_rejeitados",
    "dbo.gold_fact_faturacao",
    "dbo.gold_fact_pedido",
    "dbo.gold_fact_pedido_rejeitados"
]

print("VALIDAÇÃO FINAL DA CAMADA GOLD")
print("=" * 40)

for tabela in tabelas_gold:
    total = spark.table(tabela).count()
    print(f"{tabela}: {total}")
